In [0]:
-- 1. Read, Parse, Union, Type, Trim, and Inject promotion_id
CREATE TEMPORARY STREAMING TABLE stg_transactions_typed AS
WITH offline_stream AS (
  -- Extract only the specific columns needed from the offline table
  SELECT
    TRIM(transaction_date) AS raw_transaction_date,
    TRIM(stock_date) AS raw_stock_date,
    TRIM(product_id) AS raw_product_id,
    TRIM(customer_id) AS raw_customer_id,
    TRIM(store_id) AS raw_store_id,
    TRIM(quantity) AS raw_quantity,
    TRIM(transaction_mode) AS transaction_mode,
    ingested_timestamp
  FROM STREAM(mavenmarket.bronze.transactions)
),
online_parsed AS (
  -- Parse the Kafka JSON payload into a structured format
  SELECT 
    from_json(
      message_payload, 
      'transaction_date STRING, stock_date STRING, product_id STRING, customer_id STRING, store_id STRING, quantity STRING'
    ) AS payload,
    event_timestamp AS ingested_timestamp
  FROM STREAM(mavenmarket.bronze.kafka_online_orders)
),
online_stream AS (
  -- Extract the parsed JSON fields to match the offline schema exactly
  SELECT
    TRIM(payload.transaction_date) AS raw_transaction_date,
    TRIM(payload.stock_date) AS raw_stock_date,
    TRIM(payload.product_id) AS raw_product_id,
    TRIM(payload.customer_id) AS raw_customer_id,
    TRIM(payload.store_id) AS raw_store_id,
    TRIM(payload.quantity) AS raw_quantity,
    'online' AS transaction_mode,
    ingested_timestamp
  FROM online_parsed
),
combined_transactions AS (
  -- Now the UNION ALL is perfectly aligned (same columns, same order, same types)
  SELECT * FROM offline_stream
  UNION ALL
  SELECT * FROM online_stream
)
SELECT
  -- Generate a Surrogate Key for the transaction event itself
  md5(concat_ws('|', raw_product_id, raw_customer_id, raw_store_id, CAST(ingested_timestamp AS STRING))) AS transaction_sk,

  -- Handle date formatting anomalies
  COALESCE(
    try_to_date(raw_transaction_date, 'M/d/yyyy'),
    try_to_date(raw_transaction_date, 'yyyy-M-d'),
    try_to_date(raw_transaction_date, 'd-MMM-yy'),
    CAST(raw_transaction_date AS DATE)
  ) AS transaction_date,
  
  COALESCE(
    try_to_date(raw_stock_date, 'M/d/yyyy'),
    try_to_date(raw_stock_date, 'yyyy-M-d'),
    try_to_date(raw_stock_date, 'd-MMM-yy'),
    CAST(raw_stock_date AS DATE)
  ) AS stock_date,
  
  CAST(raw_product_id AS INT) AS product_id,
  CAST(raw_customer_id AS INT) AS customer_id,
  CAST(raw_store_id AS INT) AS store_id,
  CAST(raw_quantity AS INT) AS quantity,
  transaction_mode,
  
  -- Inject random promotion_id (0 to 20, or NULL) to link with promotions table
  CASE 
    WHEN rand() > 0.3 THEN CAST(rand() * 20 AS INT) 
    ELSE NULL 
  END AS promotion_id,
  
  ingested_timestamp
FROM combined_transactions;

-- 2. Run KPI Checks & ACTIVE SK LOOKUPS
CREATE TEMPORARY STREAMING TABLE stg_transactions_checked AS
SELECT 
  t.*,
  
  -- Pull in the Surrogate Keys from the active SCD Type 2 dimensions
  p.product_sk,
  c.customer_sk,
  s.store_sk,
  pr.promotion_sk,
  
  -- HARD CHECKS (Route to Quarantine)
  CASE WHEN t.transaction_date IS NULL THEN 'Completeness: transaction_date is null; ' ELSE '' END AS err_date_null,
  CASE WHEN t.quantity IS NULL THEN 'Completeness: quantity is null; ' ELSE '' END AS err_qty_null,
  CASE WHEN t.quantity <= 0 THEN 'Validity: quantity must be greater than 0; ' ELSE '' END AS err_qty_neg,
  CASE WHEN t.product_id IS NULL THEN 'Completeness: product_id is null; ' ELSE '' END AS err_product_null,
  CASE WHEN t.customer_id IS NULL THEN 'Completeness: customer_id is null; ' ELSE '' END AS err_customer_null,
  CASE WHEN t.store_id IS NULL THEN 'Completeness: store_id is null; ' ELSE '' END AS err_store_null,
  CASE WHEN t.transaction_mode IS NULL THEN 'Completeness: transaction_mode is null; ' ELSE '' END AS err_mode_null,
  
  -- Referential Integrity: Checked against the returned SKs
  CASE WHEN t.product_id IS NOT NULL AND p.product_sk IS NULL THEN 'Referential Integrity: product not active; ' ELSE '' END AS err_fk_product,
  CASE WHEN t.customer_id IS NOT NULL AND c.customer_sk IS NULL THEN 'Referential Integrity: customer not active; ' ELSE '' END AS err_fk_customer,
  CASE WHEN t.store_id IS NOT NULL AND s.store_sk IS NULL THEN 'Referential Integrity: store not active; ' ELSE '' END AS err_fk_store,
  CASE WHEN t.promotion_id IS NOT NULL AND pr.promotion_sk IS NULL THEN 'Referential Integrity: promotion not active; ' ELSE '' END AS err_fk_promo,
         
  -- WARN CHECKS (Stay in Silver, added to dq_flags)
  CASE WHEN t.stock_date IS NULL THEN 'Completeness: stock_date is null; ' ELSE '' END AS warn_stock_date,
  CASE WHEN t.transaction_date < t.stock_date THEN 'Consistency: transaction_date is before stock_date; ' ELSE '' END AS warn_date_logic
FROM STREAM(LIVE.stg_transactions_typed) t

-- Join to active Silver Products (Current version only)
LEFT JOIN LIVE.silver_products p 
  ON t.product_id = p.product_id 
  AND p.__END_AT IS NULL

-- Join to active Silver Customers (Current version only)
LEFT JOIN LIVE.silver_customers c 
  ON t.customer_id = c.customer_id 
  AND c.__END_AT IS NULL

-- Join to active Silver Stores (Current version only)
LEFT JOIN LIVE.silver_stores s 
  ON t.store_id = s.store_id 
  AND s.__END_AT IS NULL

-- Join to active Silver Promotions (Current version only)
LEFT JOIN LIVE.silver_promotions pr 
  ON t.promotion_id = pr.promotion_id 
  AND pr.__END_AT IS NULL;

-- 3. Consolidate Flags
CREATE TEMPORARY STREAMING TABLE stg_transactions_flagged AS
SELECT 
  *,
  TRIM(CONCAT(err_date_null, err_qty_null, err_qty_neg, err_product_null, err_customer_null, err_store_null, err_mode_null,
              err_fk_product, err_fk_customer, err_fk_store, err_fk_promo)) AS hard_failures,
  TRIM(CONCAT(warn_stock_date, warn_date_logic)) AS dq_flags
FROM STREAM(LIVE.stg_transactions_checked);

-- 4. Target Silver Table (Valid Records)
CREATE STREAMING TABLE silver_transactions
COMMENT "Cleaned transactions fact table with Surrogate Keys and warn-level data quality flags."
AS SELECT 
  * EXCEPT(err_date_null, err_qty_null, err_qty_neg, err_product_null, err_customer_null, err_store_null, err_mode_null,
           err_fk_product, err_fk_customer, err_fk_store, err_fk_promo, warn_stock_date, warn_date_logic, hard_failures)
FROM STREAM(LIVE.stg_transactions_flagged)
WHERE hard_failures = '';

-- 5. Target Quarantine Table (Invalid Records)
CREATE STREAMING TABLE quarantine_transactions
COMMENT "Quarantined transactions failing hard checks"
AS SELECT 
  * EXCEPT(err_date_null, err_qty_null, err_qty_neg, err_product_null, err_customer_null, err_store_null, err_mode_null,
           err_fk_product, err_fk_customer, err_fk_store, err_fk_promo, warn_stock_date, warn_date_logic, dq_flags),
  hard_failures AS quarantine_reasons
FROM STREAM(LIVE.stg_transactions_flagged)
WHERE hard_failures != '';